# Práctica 1 — Aprendizaje Automático 2025-26
## Notebook de Predicciones: Competición
**Autores:** Adam Kowalczyk Holtsova - Roberto Soriano Diez  
**NIAs:** 100525023 - 100522222

Este notebook carga el modelo final entrenado (`modelo_final.joblib`) y genera las predicciones sobre el dataset de competición (`bank_competition_05.pkl`), guardándolas en `predicciones.csv`.

## 1. Importaciones

In [1]:
import pandas as pd
import numpy as np
import joblib

semilla = 100525023
np.random.seed(semilla)
print('Semilla fijada:', semilla)

Semilla fijada: 100525023


## 2. Cargar el modelo final

In [2]:
modelo = joblib.load('modelo_final.joblib')
print('Modelo cargado correctamente.')
print(type(modelo))

Modelo cargado correctamente.
<class 'sklearn.pipeline.Pipeline'>


## 3. Cargar el dataset de competición


In [3]:
ruta_competicion = '../bank_ALL/bank_competition.pkl'

df_comp = pd.read_pickle(ruta_competicion)
print(f'Dataset de competición cargado: {df_comp.shape[0]} filas x {df_comp.shape[1]} columnas')
print(f'Columnas: {list(df_comp.columns)}')
df_comp.head(3)

Dataset de competición cargado: 162 filas x 16 columnas
Columnas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown


## 4. Preproceso de `pdays`

Aplicamos exactamente el mismo preproceso que en el notebook principal (sección 2.7):  
- Crear la variable binaria `was_contacted_before` (1 si pdays ≠ −1, 0 si pdays = −1).  
- Sustituir pdays = −1 por `NaN` (valor faltante real), que el pipeline ya sabe imputar.

In [4]:
df_comp = df_comp.copy()

# Variable binaria: ¿fue contactado en campaña anterior?
df_comp['was_contacted_before'] = (df_comp['pdays'] != -1).astype(int)

# pdays = -1 → NaN (sin contacto previo)
df_comp['pdays'] = df_comp['pdays'].replace(-1, np.nan)

print('Preproceso de pdays aplicado.')
print(f"  was_contacted_before=1: {df_comp['was_contacted_before'].sum()} instancias")
print(f"  pdays NaN: {df_comp['pdays'].isna().sum()} instancias")

Preproceso de pdays aplicado.
  was_contacted_before=1: 41 instancias
  pdays NaN: 121 instancias


## 5. Generar predicciones

In [5]:
predicciones = modelo.predict(df_comp)

dist = pd.Series(predicciones).value_counts()
dist_pct = pd.Series(predicciones).value_counts(normalize=True) * 100

resumen = pd.DataFrame({
    'Clase predicha': dist.index,
    'Cantidad': dist.values,
    'Porcentaje (%)': dist_pct.values.round(2)
})
print(f'Total de predicciones: {len(predicciones)}')
display(resumen)

Total de predicciones: 162


,Clase predicha,Cantidad,Porcentaje (%)
0,no,96,59.26
1,yes,66,40.74


## 6. Guardar predicciones en `predicciones.csv`

In [6]:
df_pred = pd.DataFrame({'prediccion': predicciones})
df_pred.to_csv('predicciones.csv', index=False)

print('predicciones.csv guardado correctamente.')
df_pred.head(10)

predicciones.csv guardado correctamente.


,prediccion
0,no
1,yes
2,no
3,yes
4,yes
5,no
6,yes
7,no
8,yes
9,no


## 7. Verificación: coherencia de la distribución

Comparamos la tasa de `deposit=yes` en el dataset de entrenamiento con la de las predicciones sobre competición. Una desviación pequeña (< 10 pp) indica que el modelo generaliza bien.

In [7]:
# Tasa aproximada en train (obtenida del notebook principal)
# Si tienes el pickle de y_completo puedes cargarla; aquí usamos el valor conocido
tasa_yes_train = 47.66  # % deposit=yes en el dataset completo (train+test)

tasa_yes_comp = (pd.Series(predicciones) == 'yes').mean() * 100

check = pd.DataFrame({
    'Conjunto':              ['Train completo (conocido)', 'Competición (predicho)'],
    'Tasa deposit=yes (%)':  [round(tasa_yes_train, 2),    round(tasa_yes_comp, 2)]
})
display(check)

desviacion = abs(tasa_yes_train - tasa_yes_comp)
if desviacion < 10:
    print(f' Distribuciones coherentes (desviación: {desviacion:.2f} pp).')
else:
    print(f' Desviación grande ({desviacion:.2f} pp): revisar.')

,Conjunto,Tasa deposit=yes (%)
0,Train completo (conocido),47.66
1,Competición (predicho),40.74


 Distribuciones coherentes (desviación: 6.92 pp).


## 8. Verificación Streamlit vs Pipeline

Las dos instancias de verificación definidas en `mystreamlit.py` se replican aquí para confirmar que las predicciones del pipeline y las de la app Streamlit coinciden.

In [8]:
def preparar_instancia(age, job, marital, education, default, balance,
                       housing, loan, contact, day_of_week, month, duration,
                       campaign, pdays, previous, poutcome):
    """Preproceso idéntico al de mystreamlit.py."""
    was_contacted_before = int(pdays != -1)
    pdays_proc = float('nan') if pdays == -1 else float(pdays)
    data = {
        'age': age, 'job': job, 'marital': marital, 'education': education,
        'default': default, 'balance': balance, 'housing': housing, 'loan': loan,
        'contact': contact, 'day': day_of_week, 'month': month, 'duration': duration,
        'campaign': campaign, 'pdays': pdays_proc, 'previous': previous,
        'poutcome': poutcome, 'was_contacted_before': was_contacted_before,
    }
    return pd.DataFrame([data])

# Instancia 1
inst1 = preparar_instancia(
    age=45, job='management', marital='married', education='tertiary',
    default='no', balance=5000, housing='yes', loan='no',
    contact='cellular', day_of_week=15, month='may', duration=400,
    campaign=2, pdays=-1, previous=0, poutcome='unknown'
)
pred1 = modelo.predict(inst1)[0]

# Instancia 2
inst2 = preparar_instancia(
    age=30, job='student', marital='single', education='secondary',
    default='no', balance=200, housing='no', loan='no',
    contact='telephone', day_of_week=8, month='nov', duration=120,
    campaign=5, pdays=90, previous=1, poutcome='failure'
)
pred2 = modelo.predict(inst2)[0]

print(f'Instancia 1 → Predicción pipeline: {pred1}')
print(f'Instancia 2 → Predicción pipeline: {pred2}')
print()
print('Estas predicciones deben coincidir con las mostradas en la app Streamlit.')
print('Ver capturas de pantalla en el PDF adjunto.')

Instancia 1 → Predicción pipeline: yes
Instancia 2 → Predicción pipeline: no

Estas predicciones deben coincidir con las mostradas en la app Streamlit.
Ver capturas de pantalla en el PDF adjunto.
